<a href="https://colab.research.google.com/github/zencolab/colab/blob/main/comfyui_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

安装 ComfyUI 与依赖环境

In [ ]:
# 安装ComfyUI与依赖环境
%cd /content
# 1. 拉取 ComfyUI 官方代码
!git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI

# 2. 安装核心依赖与 Hugging Face 高速下载插件
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q huggingface_hub[cli,hf_transfer]
print("✅ 环境与依赖安装完成！")

In [ ]:
#从Hugging Face满速同步模型（调用 Secrets）
import os
from google.colab import userdata

# 1. 从 Colab Secrets 中静默读取 HF Token
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("❌ 错误：请先在左侧 '🔑 Secrets' 中添加名为 'HF_TOKEN' 的密钥！")
    raise

# ⚠️ 【必填】替换为你实际的 HF 仓库地址 (例如 "username/my-comfyui-models")
REPO_ID = "username/your_model_repo"

# 2. 写入环境变量并启用 HF_TRANSFER 满速下载魔法
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("\n⏳ 正在从 Hugging Face 满速同步模型...")
# 将仓库内容同步到 ComfyUI 的模型文件夹中
!huggingface-cli download {REPO_ID} --local-dir /content/ComfyUI/models/ --repo-type model
print("✅ 模型同步完毕！")

In [ ]:
#配置 FRP 穿透并启动 ComfyUI（调用 Secrets）
import os
import subprocess
import threading
import time
import socket
from google.colab import userdata

# 1. 从 Colab Secrets 中静默读取服务器 IP 和 FRP 密码
try:
    VPS_IP = userdata.get('VPS_IP')
    FRP_TOKEN = userdata.get('FRP_TOKEN')
except userdata.SecretNotFoundError:
    print("❌ 错误：请先在左侧 '🔑 Secrets' 中添加 'VPS_IP' 和 'FRP_TOKEN'！")
    raise

# 2. 下载并解压 FRP 客户端
!wget -q https://github.com/fatedier/frp/releases/download/v0.56.0/frp_0.56.0_linux_amd64.tar.gz
!tar -zxvf frp_0.56.0_linux_amd64.tar.gz
!mv frp_0.56.0_linux_amd64/frpc /content/frpc
!chmod +x /content/frpc

# 3. 在内存中动态生成配置并写入
frpc_config = f"""
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.method = "token"
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui-colab"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8080
"""

with open('/content/frpc.toml', 'w') as f:
    f.write(frpc_config)

# 4. 定义后台监听并启动 FRP 的逻辑
def start_frpc(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            sock.close()
            break
        sock.close()

    print("\n" + "="*60)
    print("✅ ComfyUI 已启动，正在唤起 FRP 内网穿透...")
    print(f"👉 请通过浏览器访问: http://{VPS_IP}:8080")
    print("="*60 + "\n")

    subprocess.Popen(["/content/frpc", "-c", "/content/frpc.toml"])

# 5. 在后台开启穿透线程
threading.Thread(target=start_frpc, daemon=True, args=(8188,)).start()

# 6. 启动 ComfyUI 主程序
%cd /content/ComfyUI
!python main.py --dont-print-server